In [189]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.ensemble import VotingClassifier, VotingRegressor, BaggingRegressor
from sklearn.metrics import classification_report, f1_score, accuracy_score, log_loss, r2_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import make_column_selector,make_column_transformer

In [7]:
sonar=pd.read_csv('Sonar.csv')
y=sonar['Class']
X=sonar.drop('Class',axis=1)
le=LabelEncoder()
y=le.fit_transform(y)

In [8]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=25,stratify=y)

In [9]:
dtc=DecisionTreeClassifier(random_state=25)
knn=KNeighborsClassifier()
nb=GaussianNB()
voting=VotingClassifier(estimators=[('TREE',dtc),('KNN',knn),('NB',nb)])
voting.fit(X_train,y_train)
y_pred=voting.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.67      0.85      0.75        34
           1       0.75      0.52      0.61        29

    accuracy                           0.70        63
   macro avg       0.71      0.69      0.68        63
weighted avg       0.71      0.70      0.69        63



In [23]:
dtc1=DecisionTreeClassifier(random_state=25)
dtc2=DecisionTreeClassifier(random_state=25,max_depth=3)
knn1=KNeighborsClassifier()
knn2=KNeighborsClassifier(n_neighbors=3)
nb=GaussianNB()
voting=VotingClassifier(estimators=[('TREE1',dtc1),('TREE2',dtc2),('KNN1',knn1),('KNN2',knn2),('NB',nb)])
voting.fit(X_train,y_train)
y_pred=voting.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.67      0.85      0.75        34
           1       0.75      0.52      0.61        29

    accuracy                           0.70        63
   macro avg       0.71      0.69      0.68        63
weighted avg       0.71      0.70      0.69        63



## Evaluating individual estimators

In [24]:
y_pred_0=voting.estimators_[0].predict(X_test)
y_pred_1=voting.estimators_[1].predict(X_test)
y_pred_2=voting.estimators_[2].predict(X_test)
y_pred_3=voting.estimators_[3].predict(X_test)
y_pred_4=voting.estimators_[4].predict(X_test)

In [25]:
for i in range(len(voting.estimators_)):
    print('Estimator: ',voting.estimators_[i])
    print('Accuracy Score: ',accuracy_score(y_test,voting.estimators_[i].predict(X_test)))

Estimator:  DecisionTreeClassifier(random_state=25)
Accuracy Score:  0.6984126984126984
Estimator:  DecisionTreeClassifier(max_depth=3, random_state=25)
Accuracy Score:  0.6666666666666666
Estimator:  KNeighborsClassifier()
Accuracy Score:  0.746031746031746
Estimator:  KNeighborsClassifier(n_neighbors=3)
Accuracy Score:  0.8095238095238095
Estimator:  GaussianNB()
Accuracy Score:  0.6349206349206349


## Soft Voting

In [27]:
dtc1=DecisionTreeClassifier(random_state=25)
dtc2=DecisionTreeClassifier(random_state=25,max_depth=3)
knn1=KNeighborsClassifier()
knn2=KNeighborsClassifier(n_neighbors=3)
nb=GaussianNB()
voting=VotingClassifier(estimators=[('TREE1',dtc1),('TREE2',dtc2),('KNN1',knn1),('KNN2',knn2),('NB',nb)],voting='soft')
voting.fit(X_train,y_train)
y_pred=voting.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.68      0.88      0.77        34
           1       0.79      0.52      0.62        29

    accuracy                           0.71        63
   macro avg       0.74      0.70      0.70        63
weighted avg       0.73      0.71      0.70        63



## Specifying Weights

In [29]:
dtc1=DecisionTreeClassifier(random_state=25)
dtc2=DecisionTreeClassifier(random_state=25,max_depth=3)
knn1=KNeighborsClassifier()
knn2=KNeighborsClassifier(n_neighbors=3)
nb=GaussianNB()
voting=VotingClassifier(estimators=[('TREE1',dtc1),('TREE2',dtc2),('KNN1',knn1),('KNN2',knn2),('NB',nb)],voting='soft',weights=[7,6,7.5,8.1,6.3])
voting.fit(X_train,y_train)
y_pred=voting.predict(X_test)
print(classification_report(y_test,y_pred))
y_pred_prob = voting.predict_proba(X_test)
print(log_loss(y_test,y_pred_prob))

              precision    recall  f1-score   support

           0       0.68      0.88      0.77        34
           1       0.79      0.52      0.62        29

    accuracy                           0.71        63
   macro avg       0.74      0.70      0.70        63
weighted avg       0.73      0.71      0.70        63

0.4792459136222534


## Voting on HR Database

In [72]:
hr=pd.read_csv('HR_comma_sep.csv')
y=hr['left']
X=hr.drop('left',axis=1)
le=LabelEncoder()
y=le.fit_transform(y)
ohe=OneHotEncoder(sparse_output=False,drop='first')

In [73]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=25,stratify=y)
X_train=ohe.fit_transform(X_train)
X_test=ohe.transform(X_test)

In [74]:
dtc1=DecisionTreeClassifier(random_state=25)
dtc2=DecisionTreeClassifier(random_state=25,max_depth=3)
knn1=KNeighborsClassifier()
knn2=KNeighborsClassifier(n_neighbors=3)
nb=GaussianNB()
voting=VotingClassifier(estimators=[('TREE1',dtc1),('TREE2',dtc2),('KNN1',knn1),('KNN2',knn2),('NB',nb)])
voting.fit(X_train,y_train)
y_pred=voting.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      0.94      0.96      3429
           1       0.84      0.93      0.88      1070

    accuracy                           0.94      4499
   macro avg       0.91      0.93      0.92      4499
weighted avg       0.94      0.94      0.94      4499



In [75]:
y_pred_0=voting.estimators_[0].predict(X_test)
y_pred_1=voting.estimators_[1].predict(X_test)
y_pred_2=voting.estimators_[2].predict(X_test)
y_pred_3=voting.estimators_[3].predict(X_test)
y_pred_4=voting.estimators_[4].predict(X_test)

In [76]:
for i in range(len(voting.estimators_)):
    print('Estimator: ',voting.estimators_[i])
    print('Accuracy Score: ',accuracy_score(y_test,voting.estimators_[i].predict(X_test)))

Estimator:  DecisionTreeClassifier(random_state=25)
Accuracy Score:  0.9624360969104245
Estimator:  DecisionTreeClassifier(max_depth=3, random_state=25)
Accuracy Score:  0.8395198933096244
Estimator:  KNeighborsClassifier()
Accuracy Score:  0.914203156256946
Estimator:  KNeighborsClassifier(n_neighbors=3)
Accuracy Score:  0.9242053789731052
Estimator:  GaussianNB()
Accuracy Score:  0.7457212713936431


In [77]:
dtc1=DecisionTreeClassifier(random_state=25)
dtc2=DecisionTreeClassifier(random_state=25,max_depth=3)
knn1=KNeighborsClassifier()
knn2=KNeighborsClassifier(n_neighbors=3)
nb=GaussianNB()
voting=VotingClassifier(estimators=[('TREE1',dtc1),('TREE2',dtc2),('KNN1',knn1),('KNN2',knn2),('NB',nb)],voting='soft')
voting.fit(X_train,y_train)
y_pred=voting.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.99      0.96      0.97      3429
           1       0.88      0.95      0.91      1070

    accuracy                           0.96      4499
   macro avg       0.93      0.96      0.94      4499
weighted avg       0.96      0.96      0.96      4499



In [78]:
dtc1=DecisionTreeClassifier(random_state=25)
dtc2=DecisionTreeClassifier(random_state=25,max_depth=3)
knn1=KNeighborsClassifier()
knn2=KNeighborsClassifier(n_neighbors=3)
nb=GaussianNB()
voting=VotingClassifier(estimators=[('TREE1',dtc1),('TREE2',dtc2),('KNN1',knn1),('KNN2',knn2),('NB',nb)],voting='soft',weights=[9.6,8.3,9.1,9.2,7.4])
voting.fit(X_train,y_train)
y_pred=voting.predict(X_test)
print(classification_report(y_test,y_pred))
y_pred_prob = voting.predict_proba(X_test)
print(log_loss(y_test,y_pred_prob))

              precision    recall  f1-score   support

           0       0.99      0.96      0.97      3429
           1       0.88      0.95      0.91      1070

    accuracy                           0.96      4499
   macro avg       0.93      0.96      0.94      4499
weighted avg       0.96      0.96      0.96      4499

0.17717211361248844


## Concrete Strength (Using Voting Regressor)

In [108]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.ensemble import VotingClassifier,VotingRegressor
from sklearn.metrics import classification_report,f1_score,accuracy_score,log_loss,r2_score
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.compose import make_column_selector,make_column_transformer
from sklearn.linear_model import LinearRegression, ElasticNet, LogisticRegression

In [109]:
concrete=pd.read_csv('Concrete_Data.csv')
y=concrete['Strength']
X=concrete.drop('Strength',axis=1)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=25)
ss=StandardScaler()
# X_train=ss.fit_transform(X_train)
# X_test=ss.transform(X_test)

In [118]:
lr=LinearRegression()
elnet=ElasticNet()
dtr1=DecisionTreeRegressor(random_state=25)
dtr2=DecisionTreeRegressor(random_state=25,max_depth=5)
voting=VotingRegressor(estimators=[('Linear Regression',lr),('ElasticNet',elnet),('Decision Tree 1',dtr1),('Decision Tree 2',dtr2)])
voting.fit(X_train,y_train)
y_pred=voting.predict(X_test)
print(r2_score(y_test,y_pred))

0.8260276834012175


In [120]:
y_pred_0=voting.estimators_[0].predict(X_test)
y_pred_1=voting.estimators_[1].predict(X_test)
y_pred_2=voting.estimators_[2].predict(X_test)
y_pred_3=voting.estimators_[3].predict(X_test)
for i in range(len(voting.estimators_)):
    print('Estimator: ',voting.estimators_[i])
    print('R2 Score: ',r2_score(y_test,voting.estimators_[i].predict(X_test)))

Estimator:  LinearRegression()
R2 Score:  0.6351839142464111
Estimator:  ElasticNet()
R2 Score:  0.6345321364921961
Estimator:  DecisionTreeRegressor(random_state=25)
R2 Score:  0.8127760533837747
Estimator:  DecisionTreeRegressor(max_depth=5, random_state=25)
R2 Score:  0.7311101169515943


In [147]:
voting=VotingRegressor(estimators=[('Linear Regression',lr),('ElasticNet',elnet),('Decision Tree 1',dtr1),('Decision Tree 2',dtr2)],weights=[2,3,10,6])
voting.fit(X_train,y_train)
y_pred=voting.predict(X_test)
print(r2_score(y_test,y_pred))

0.8535877300239261


## Bagging Classifier

In [174]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.ensemble import VotingClassifier, VotingRegressor, BaggingClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score, log_loss, r2_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.compose import make_column_selector, make_column_transformer
from tqdm import tqdm

sonar = pd.read_csv('Sonar.csv')
y = sonar['Class']
X = sonar.drop('Class', axis=1)
le = LabelEncoder()
y = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25, stratify=y)


In [175]:
dtc = DecisionTreeClassifier(random_state=25)
knn = KNeighborsClassifier()
nb = GaussianNB()
bagg = BaggingClassifier(estimator=nb,n_estimators=10,random_state=25)
bagg.fit(X_train, y_train)
y_pred = bagg.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.62      0.74      0.68        34
           1       0.61      0.48      0.54        29

    accuracy                           0.62        63
   macro avg       0.62      0.61      0.61        63
weighted avg       0.62      0.62      0.61        63



In [176]:
dtc = DecisionTreeClassifier(random_state=25)
knn = KNeighborsClassifier()
nb = GaussianNB()
bagg = BaggingClassifier(estimator=nb,n_estimators=10,bootstrap=True,random_state=25)
bagg.fit(X_train, y_train)
y_pred = bagg.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.62      0.74      0.68        34
           1       0.61      0.48      0.54        29

    accuracy                           0.62        63
   macro avg       0.62      0.61      0.61        63
weighted avg       0.62      0.62      0.61        63



In [182]:
lr=LogisticRegression()
est_list=[nb,dtc,knn,lr]
n_est=[10,15,25,50]
scores=[]
for e in tqdm(est_list):
        for n in n_est:
            bagg=BaggingClassifier(estimator=e,n_estimators=n,random_state=25)
            bagg.fit(X_train, y_train)
            y_pred_prob=bagg.predict_proba(X_test)
            scores.append([e,n,log_loss(y_test,y_pred_prob)])
df_scores=pd.DataFrame(scores,columns=['Estimator','B Samples','Score'])
df_scores.sort_values('Score')

100%|██████████| 4/4 [00:02<00:00,  1.46it/s]


,Estimator,B Samples,Score
8,KNeighborsClassifier(),10,0.422282
9,KNeighborsClassifier(),15,0.431490
10,KNeighborsClassifier(),25,0.432707
11,KNeighborsClassifier(),50,0.447037
6,DecisionTreeClassifier(random_state=25),25,0.475750
4,DecisionTreeClassifier(random_state=25),10,0.477146
7,DecisionTreeClassifier(random_state=25),50,0.490503
5,DecisionTreeClassifier(random_state=25),15,0.502120
13,LogisticRegression(),15,0.534413
12,LogisticRegression(),10,0.537041


## For the same estimator tuning

In [183]:
depths = [None, 3,5,7]
scores = []
for d in tqdm(depths):
    dtc = DecisionTreeClassifier(random_state=25, max_depth=d)
    bagg=BaggingClassifier(estimator=dtc,n_estimators=50,random_state=25)
    bagg.fit(X_train, y_train)
    y_pred_prob=bagg.predict_proba(X_test)
    scores.append([d,log_loss(y_test,y_pred_prob)])
df_scores=pd.DataFrame(scores,columns=['depth','score'])
df_scores.sort_values('score')

100%|██████████| 4/4 [00:01<00:00,  3.69it/s]


,depth,score
0,NaN,0.490503
3,7.0,0.492312
2,5.0,0.495034
1,3.0,0.525085


In [190]:
dtc = DecisionTreeRegressor(random_state=25)
knn = KNeighborsRegressor()
lr = LinearRegression()

In [193]:
est_list = [dtc, knn, lr]
n_est=[10,15,25,50]
scores=[]
for e in tqdm(est_list):
    for n in n_est:
        bagg=BaggingRegressor(estimator=e,n_estimators=n,random_state=25)
        bagg.fit(X_train, y_train)
        y_pred=bagg.predict(X_test)
        scores.append([e, n, r2_score(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=['Estimator','B Samples','Score'])
df_scores.sort_values('Score', ascending=False)

100%|██████████| 3/3 [00:01<00:00,  1.92it/s]


,Estimator,B Samples,Score
4,KNeighborsRegressor(),10,0.428093
5,KNeighborsRegressor(),15,0.414703
6,KNeighborsRegressor(),25,0.409717
7,KNeighborsRegressor(),50,0.387783
0,DecisionTreeRegressor(random_state=25),10,0.352110
2,DecisionTreeRegressor(random_state=25),25,0.350729
3,DecisionTreeRegressor(random_state=25),50,0.337286
1,DecisionTreeRegressor(random_state=25),15,0.322718
10,LinearRegression(),25,0.141168
11,LinearRegression(),50,0.139639


In [194]:
depths = [None, 3,5,7]
scores = []
for d in tqdm(depths):
    dtc = DecisionTreeRegressor(random_state=25, max_depth=d)
    bagg=BaggingRegressor(estimator=dtc,n_estimators=50,random_state=25)
    bagg.fit(X_train, y_train)
    y_pred=bagg.predict(X_test)
    scores.append([d,r2_score(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=['depth','score'])
df_scores.sort_values('score', ascending=False)

100%|██████████| 4/4 [00:00<00:00,  4.31it/s]


,depth,score
0,NaN,0.337286
3,7.0,0.333972
2,5.0,0.333387
1,3.0,0.276335


In [195]:
concrete=pd.read_csv('Concrete_Data.csv')
y=concrete['Strength']
X=concrete.drop('Strength',axis=1)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=25)

In [197]:
depths = [None, 3,5,7]
scores = []
for d in tqdm(depths):
    dtc = DecisionTreeRegressor(random_state=25, max_depth=d)
    bagg=BaggingRegressor(estimator=dtc,n_estimators=50,random_state=25)
    bagg.fit(X_train, y_train)
    y_pred=bagg.predict(X_test)
    scores.append([d,r2_score(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=['depth','score'])
df_scores.sort_values('score')

100%|██████████| 4/4 [00:00<00:00,  4.90it/s]


,depth,score
1,3.0,0.643232
2,5.0,0.801792
3,7.0,0.856205
0,NaN,0.880861


In [198]:
dtc = DecisionTreeRegressor(random_state=25)
knn = KNeighborsRegressor()
lr = LinearRegression()

In [199]:
est_list = [dtc, knn, lr]
n_est=[10,15,25,50]
scores=[]
for e in tqdm(est_list):
    for n in n_est:
        bagg=BaggingRegressor(estimator=e,n_estimators=n,random_state=25)
        bagg.fit(X_train, y_train)
        y_pred=bagg.predict(X_test)
        scores.append([e, n, r2_score(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=['Estimator','B Samples','Score'])
df_scores.sort_values('Score', ascending=False)

100%|██████████| 3/3 [00:01<00:00,  1.89it/s]


,Estimator,B Samples,Score
2,DecisionTreeRegressor(random_state=25),25,0.881644
3,DecisionTreeRegressor(random_state=25),50,0.880861
1,DecisionTreeRegressor(random_state=25),15,0.877123
0,DecisionTreeRegressor(random_state=25),10,0.876503
7,KNeighborsRegressor(),50,0.712140
6,KNeighborsRegressor(),25,0.708045
5,KNeighborsRegressor(),15,0.702541
4,KNeighborsRegressor(),10,0.696449
8,LinearRegression(),10,0.634298
9,LinearRegression(),15,0.634216


In [202]:
depths = [None, 3,5,7]
scores = []
for d in tqdm(depths):
    dtc = DecisionTreeRegressor(random_state=25, max_depth=d)
    bagg=BaggingRegressor(estimator=dtc,n_estimators=50,random_state=25)
    bagg.fit(X_train, y_train)
    y_pred=bagg.predict(X_test)
    scores.append([d,r2_score(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=['depth','score'])
df_scores.sort_values('score', ascending=False)

100%|██████████| 4/4 [00:00<00:00,  5.04it/s]


,depth,score
0,NaN,0.880861
3,7.0,0.856205
2,5.0,0.801792
1,3.0,0.643232


In [203]:
dtc = DecisionTreeRegressor(random_state=25, max_depth=7)
bagg=BaggingRegressor(estimator=dtc,n_estimators=50,random_state=25, oob_score=True)
bagg.fit(X_train, y_train)
print("oob score:", bagg.oob_score_)

oob score: 0.8733468463801077
